# One plan, every fan-out — the model stage writes the same bytes at any width

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/26-fanout/fanout.ipynb)

Built from [`cookbook/book/chapters/26-fanout/fanout.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/26-fanout/fanout.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `generate_embeddings` under `[inference] partitions = 1, 2, 4` ·
`describe_table` (`artifact`, `definition_hash`) · **Theory:** a parallel plan
is deterministic when the unit of work is fixed before the work is
distributed (Kleppmann 2017) · **Rail:** measurement (every verdict an
equality, frozen to golden, tolerance 0).

A model stage attached to a query engine as a user-defined function sees
whatever batches the engine hands it. How rows are grouped then depends on the
partition count and on how the exchanges above it re-batch, so a padded forward
over a different grouping produces different bytes. In jammi, the model stage
is a plan operator (`InferenceExec`), so it can fix the grouping before any
rows are distributed:

1. **Rows are costed once.** Each row's cost is its length along the axis a
   forward pads (its token count for text). The numbered input orders rows by
   cost, then key.
2. **Chunks are cut once.** The ordered rows are cut into forward chunks under
   the chunk budget (`batch_size` rows, `batch_tokens` padded tokens), and each
   row carries its chunk id.
3. **The exchange hashes on the chunk id.** A fan-out of *N* partitions
   distributes whole chunks, never dividing one. The `InferenceFanOut` rule
   keeps that exchange at the node's own declared width through DataFusion's
   own distribution enforcement.

So every partition count forwards identical chunks and writes identical bytes.
This chapter measures that: the same embedding plan over the same 24-row
corpus, run at partitions 1, 2 and 4, each in a fresh catalog.

## The corpus and the three runs

The rows are one to fourteen words long — the lengths are what the chunk cut
orders on, so they are deliberately uneven. `batch_size = 4` makes every
fan-out forward several chunks.

In [ ]:
import tempfile
from pathlib import Path

import jammi
import pyarrow as pa
import pyarrow.parquet as pq

from jammi_cookbook import contracts, fixtures

MODEL = fixtures.model("tiny_bert")
PARTITIONS = (1, 2, 4)
BATCH_SIZE = 4

docs = pa.table({
    "_row_id": [f"d{i:02d}" for i in range(24)],
    "text": [
        "graph",
        "signal processing on graphs",
        "a spectral filter applied to node features over the normalized laplacian",
        "message passing",
        "random walks with restart as a personalized ranking",
        "edge weights",
        "attention over neighbours weights each message by a learned compatibility score",
        "pooling",
        "a laplacian eigenmap embeds nodes by the smallest nontrivial eigenvectors",
        "k nearest neighbours",
        "propagation smooths a signal along the edges of the graph",
        "label spreading",
        "a node embedding",
        "the adjacency matrix and its powers count walks between node pairs",
        "diffusion",
        "graph convolution as a first order polynomial of the laplacian",
        "readout",
        "a heterogeneous graph carries typed nodes and typed edges",
        "clustering coefficient",
        "spectral clustering partitions the graph by the fiedler vector of its laplacian",
        "homophily",
        "over smoothing makes deep propagation collapse node representations together",
        "subgraph sampling",
        "the incidence matrix relates edges to their endpoints",
    ],
})


def embed_at(partitions: int) -> dict:
    """Embed the corpus in a fresh catalog fanned over `partitions`, and read
    back what the table recorded about itself."""
    with tempfile.TemporaryDirectory() as tmp:
        corpus = Path(tmp) / "docs.parquet"
        pq.write_table(docs, corpus)
        config = Path(tmp) / "jammi.toml"
        config.write_text(
            f"[inference]\npartitions = {partitions}\nbatch_size = {BATCH_SIZE}\n"
        )
        with jammi.connect(f"file://{tmp}/catalog", config=str(config)) as db:
            db.add_source("docs", url=str(corpus), format="parquet")
            table = db.generate_embeddings(
                source="docs", model=MODEL, columns=["text"], key="_row_id"
            )
            recorded = db.describe_table(table)
    return {
        "partitions": partitions,
        "artifact": recorded["artifact"],
        "definition_hash": recorded["definition_hash"],
    }


runs = [embed_at(n) for n in PARTITIONS]
for run in runs:
    print(f"partitions={run['partitions']}  artifact={run['artifact']}")

## The verdicts

`describe_table` returns a table's recorded materialization: its artifact
digest covers the table's Parquet bytes, so equal digests mean equal bytes; its
definition hash is the recorded identity of what produced the table. The
definition is the same across runs because the partition count changes how the
work is spread, not what is computed.

In [ ]:
reference = runs[0]
for run in runs[1:]:
    same = 1.0 if run["artifact"] == reference["artifact"] else 0.0
    contracts.assert_close(f"fanout.p{run['partitions']}_artifact_equal", same)
    print(f"partitions={run['partitions']}: same bytes as partitions=1 → {same}")

one_definition = 1.0 if len({r["definition_hash"] for r in runs}) == 1 else 0.0
contracts.assert_close("fanout.definition_equal_all", one_definition)
print(f"one definition across every fan-out → {one_definition}")

## Why this matters beyond one table

- **Reuse is safe at any width.** A table materialized at one fan-out is the
  cached answer for the same definition at another. The definition hash
  doesn't carry the partition count, and the bytes it names don't depend on it.
- **Verification is width-independent.** `verify_materialization` recomputes
  an artifact digest; a verifier running a different fan-out than the
  producer still gets a `match`.
- **Scaling out is a configuration change.** The same holds when the plan's
  partitions run as tasks of a Ballista cluster. The chunk ids travel with
  the rows through the shuffle, so a cluster writes what one process writes.
  The engine's distributed tests prove that leg; this chapter proves the
  in-process one.

The mechanism lives in `crates/jammi-datafusion/src/inference/` (`numbered.rs`
numbers and cuts; `exec.rs` holds `InferenceExec` and `InferenceFanOut`). The
guide's *Built on DataFusion and Ballista* page maps it among the engine's
other extensions of DataFusion.

## References

- Kleppmann, Martin (2017) *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems* O'Reilly Media.